# Friction checks — test bench

Run each piece of `friction.py` and look at what it produces. Nothing here
reaches a verdict; it reports numbers and draws pictures.

| check | question |
|---|---|
| 1 · crossing | what if a buy lifted `ask_1` and a sell hit `bid_1`? |
| 2 · fees | what if every MWh paid a fee, in 5 cent steps? |
| 3 · delay | what if fills landed late, on each strategy's own clock? |

Throughout, **mid is taken as the claim**: repricing at mid reproduces every
registry `claimed` figure to rounding, so the two are the same baseline.

In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, "../registry")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 240, "display.max_columns", 40)

from repricer import load_market, load_blotter
from loader import load_registry
import friction as fr

market   = load_market("../data")      # ~30s, the book is 524k rows
registry = load_registry("../registry")
blotters = {k: load_blotter(f"../blotters/{k}-blotter.csv") for k in registry}
print(f"{len(registry)} submissions, {sum(len(b) for b in blotters.values()):,} trades")

## 0 · Unit tests

The building blocks are tested against a hand-built book with answers you can
check by eye — no real data, so a failure points at the code, not the market.

In [ ]:
import pytest

# run the suite in-process so its output lands in this notebook,
# and so it behaves the same on Windows and POSIX
pytest.main(["-q", "test_friction.py"])

## 1 · Crossing the spread

`pnl_retained` is the share of the mid P&L that survives paying the spread.
`half_spread` is what each MWh costs to trade; `edge_mid` is what it earns.

In [ ]:
cols = ["pnl_mid", "pnl_cross", "pnl_retained", "spread_cost_eur",
        "sharpe_mid", "sharpe_cross", "sortino_mid", "sortino_cross",
        "mdd_mid", "mdd_cross", "edge_mid", "edge_cross", "half_spread"]
summary = fr.crossing_summary(registry, market)
summary[cols].sort_values("pnl_retained", ascending=False).round(3)

One submission in full — every metric the book quotes, at mid and crossed.
Change `SUB` to inspect another.

In [ ]:
SUB = "windfall"
fr.crossing_check(registry[SUB], blotters[SUB], market).round(3)

In [ ]:
for key in sorted(registry):
    fig = fr.plot_crossing(registry[key], blotters[key], market, show=True)
    display(fig)
    plt.close(fig)   # display() already rendered it; close to avoid a double

## 2 · Final performance, at the prices you would actually get

Dropping the mid comparison: this is each strategy as it would have performed
crossing the spread. Equity curve, drawdown, monthly P&L.

In [ ]:
fr.performance_table(registry, market, fill="cross").round(3)

In [ ]:
for key in sorted(registry):
    fig = fr.plot_performance(registry[key], blotters[key], market,
                              fill="cross", show=True)
    display(fig)
    plt.close(fig)

## 3 · Sensitivity to fees

A fee is charged on every MWh and always works against you, so P&L is exactly
linear in it — and the **breakeven fee is exactly the edge per MWh**. Swept in
5 cent steps on top of crossing.

In [ ]:
fees = fr.fee_sensitivity_table(registry, market)      # step=0.05 by default
fees.round(3)

In [ ]:
fig = fr.plot_fee_curves(registry, market, show=True)
display(fig); plt.close(fig)

In [ ]:
# one submission, P&L and Sharpe against the fee, breakeven marked
SUB = "blackbox"
display(fr.fee_sensitivity(registry[SUB], blotters[SUB], market).round(3))
fig = fr.plot_fee_sensitivity(registry[SUB], blotters[SUB], market, show=True)
display(fig); plt.close(fig)

## 4 · Execution delay, on each strategy's own clock

A fixed 15-minute delay means something very different to a strategy holding
for 15 minutes than to one holding for ten hours, so the delay is scaled to
each strategy's **median holding time**, measured by FIFO-matching every open
against its close. Fractions are log-spaced from 1% to 50%.

**The book only snapshots every 15 minutes, and that binds hard at the small
end.** A delay under half a snapshot (7.5 min) rounds to no delay at all, so
each strategy has a `resolution_floor` below which this check cannot say
anything: 1% for `blackbox` (holds 9.75h) but **50% for `pingpong`** (holds
15 min), which therefore gets exactly one usable point. Cells below the floor
are `NaN`, not zero, and the chart draws those points hollow — a flat run of
hollow markers means *no data*, not *no effect*.


In [ ]:
# holding times first - `unmatched_mw` should be 0: every position is closed
delays = fr.delay_summary(registry, market)
delays.round(3)

In [ ]:
fig = fr.plot_delay_sensitivity(registry, market, show=True)
display(fig); plt.close(fig)

In [ ]:
# one submission in full: what was asked for vs. what the book could apply
SUB = "windfall"
fr.delay_sensitivity(registry[SUB], blotters[SUB], market).round(3)

## 5 · Things worth knowing

- **Sortino convention.** The book's `sortino_ann` divides by the standard
  deviation of *losing days about their own mean* (`ddof=1`), not by downside
  deviation about zero. That reproduces all seven claimed values exactly, so
  it is their convention — but it is the more flattering of the two.
- **Sortino can rise under crossing** (`windfall2`). Crossing shifts which days
  are losers; if it turns a few small winners into small, similar-sized losers,
  the loss distribution tightens and the ratio goes up. An artefact of the
  denominator above, not an improvement.
- **Drawdown degrades faster than P&L.** `blackbox` keeps 35% of its P&L but
  more than doubles its worst drawdown.
- **Delay is not monotonic for everyone.** `bounceback` *makes more* money when
  filled late, which is what you would expect of a reversion strategy entering
  too early — worth a check of its own rather than a footnote.
- **The delay grid is coarse for fast strategies.** `pingpong` holds 15 minutes,
  so its first three delay steps all round to zero. Read `realised_fraction`,
  not the requested one.
- **`half_spread` is the mean over the trades a strategy actually did**, not
  over the market — a strategy that trades in wide books pays more.